In [191]:
import os
import pandas as pd
import numpy as np
import matplotlib
import networkx as nx

print(os.getcwd())

BUILDING_NAME = 'Edificio1.csv'

ASCENDING_SPEED = 1.0
DESCENDING_SPEED = 2.0
HORIZONTAL_SPEED = 1.5

ASCENDING_ENERGY= 50
DESCENDING_ENERGY= 5
HORIZONTAL_ENERGY = 10

BATTERY_CAPACITY = 3600 if 'Edificio1.csv' == BUILDING_NAME else 21600 #1 Wh is 3600 J

Y_ENTRY_POINT = -12.5 if 'Edificio1.csv' == BUILDING_NAME else -20.0

/home/michele/big-project-FOR


# Reading CSV and creating pairs


In [192]:
#loading building csv data
building = pd.read_csv(BUILDING_NAME)

#setting node numbering from 1 to n
building.index += 1 

#generating all possible pairs of points (edges)
nodes_pairs = building.rename_axis('source_node').reset_index().merge(building.rename_axis('target_node').reset_index(), how='cross')

#removing self-self pairs
nodes_pairs = nodes_pairs[nodes_pairs['source_node'] != nodes_pairs['target_node']] #keeping only pairs with different points

In [193]:
first_point_cols= ['x_x','y_x','z_x']
second_point_cols=['x_y','y_y','z_y']
diff_cols= ['dx','dy','dz']

# Computing distances 

In [194]:
#function to determine if two points are connected based on distance and coordinate deltas

def process_connections(dataframe):
    #distances and deltas on coordinates
    dataframe[diff_cols] = (dataframe[second_point_cols].values - dataframe[first_point_cols].values)
    dataframe['euclidean_distance'] = np.sqrt((dataframe[diff_cols]**2).sum(axis=1)) #axis=1 sum on columns 
    
    
    # conditions
    condition_near = (dataframe['euclidean_distance'] <= 4.0)
    
    
    condition_tunnel = (
        ((dataframe['dx'].abs() <= 0.5) & (dataframe['dy'].abs() <= 0.5)) | 
        ((dataframe['dy'].abs() <= 0.5) & (dataframe['dz'].abs() <= 0.5)) | 
        ((dataframe['dx'].abs() <= 0.5) & (dataframe['dz'].abs() <= 0.5))
    )
    condition_tunnel_and_not_far = (dataframe['euclidean_distance'] <= 11.0) & condition_tunnel
    
    # final connection condition
    dataframe["are_connected"] = condition_near | condition_tunnel_and_not_far
    return dataframe



In [195]:
nodes_pairs = process_connections(nodes_pairs)

nodes_pairs.head()

,source_node,x_x,y_x,z_x,target_node,x_y,y_y,z_y,dx,dy,dz,euclidean_distance,are_connected
1,1,2.5,3.25,0.0,2,-6.0,3.25,0.0,-8.5,0.0,0.0,8.500000,True
2,1,2.5,3.25,0.0,3,-6.0,11.75,0.0,-8.5,8.5,0.0,12.020815,False
3,1,2.5,3.25,0.0,4,2.5,11.75,0.0,0.0,8.5,0.0,8.500000,True
4,1,2.5,3.25,0.0,5,2.5,3.25,11.0,0.0,0.0,11.0,11.000000,True
5,1,2.5,3.25,0.0,6,-6.0,3.25,11.0,-8.5,0.0,11.0,13.901439,False


In [196]:
#computing vertical distances on z and distances on horizontal plane x,y
nodes_pairs["d_horizontal"] = np.sqrt(nodes_pairs['dx']**2 + nodes_pairs['dy']**2)
nodes_pairs["d_vertical"] = nodes_pairs['dz']

#dropping columns that are not useful anymore
nodes_pairs.drop(columns=['dx','dy','dz'], inplace=True)
nodes_pairs.drop(columns=['x_x','y_x','z_x','x_y','y_y','z_y'], inplace=True)

#keeping only connected pairs
nodes_pairs = nodes_pairs[nodes_pairs['are_connected']==True]
nodes_pairs.drop(columns=['are_connected'], inplace=True)

nodes_pairs.head(5)

,source_node,target_node,euclidean_distance,d_horizontal,d_vertical
1,1,2,8.500000,8.500000,0.0
3,1,4,8.500000,8.500000,0.0
4,1,5,11.000000,0.000000,11.0
18,1,19,4.007805,4.007805,0.0
19,1,20,1.030776,1.030776,0.0


# Function for time travel

In [197]:
#finding rows corresponding to different movement types 

d_horizontal = nodes_pairs['d_horizontal']
d_vertical = nodes_pairs['d_vertical']
rows_horizontal = (d_vertical == 0)
rows_ascending = (d_horizontal == 0) & (d_vertical > 0)
rows_descending = (d_horizontal == 0) & (d_vertical < 0)
rows_oblique_ascending = (d_horizontal != 0) & (d_vertical > 0)
rows_oblique_descending = (d_horizontal != 0) & (d_vertical < 0)

In [198]:
#function for time travel based on movement types and speeds

def compute_time_travel(dataframe):
    dataframe.loc[rows_horizontal, 'time_travel'] = dataframe.loc[rows_horizontal, 'd_horizontal'] / HORIZONTAL_SPEED

    dataframe.loc[rows_ascending, 'time_travel'] = dataframe.loc[rows_ascending, 'd_vertical'] / ASCENDING_SPEED
    dataframe.loc[rows_descending, 'time_travel'] = -dataframe.loc[rows_descending, 'd_vertical'] / DESCENDING_SPEED

    dataframe.loc[rows_oblique_ascending, 'time_travel'] = np.maximum(dataframe.loc[rows_oblique_ascending, 'd_horizontal'] / HORIZONTAL_SPEED,
                                                                     dataframe.loc[rows_oblique_ascending, 'd_vertical'] / ASCENDING_SPEED)
    
    dataframe.loc[rows_oblique_descending, 'time_travel'] = np.maximum(dataframe.loc[rows_oblique_descending, 'd_horizontal'] / HORIZONTAL_SPEED,
                                                                     -dataframe.loc[rows_oblique_descending, 'd_vertical'] / DESCENDING_SPEED)
    return dataframe

# Function for energy consumption


In [199]:
#function for energy cost based on movement types and energy consumption rates

def compute_energy_cost(dataframe):
    dataframe.loc[rows_horizontal, 'energy_consumption'] = dataframe.loc[rows_horizontal, 'd_horizontal'] * HORIZONTAL_ENERGY

    dataframe.loc[rows_ascending, 'energy_consumption'] = dataframe.loc[rows_ascending, 'd_vertical'] * ASCENDING_ENERGY
    dataframe.loc[rows_descending, 'energy_consumption'] = -dataframe.loc[rows_descending, 'd_vertical'] * DESCENDING_ENERGY

    dataframe.loc[rows_oblique_ascending, 'energy_consumption'] = (dataframe.loc[rows_oblique_ascending, 'd_horizontal'] * HORIZONTAL_ENERGY +
                                                                  dataframe.loc[rows_oblique_ascending, 'd_vertical'] * ASCENDING_ENERGY)
    
    dataframe.loc[rows_oblique_descending, 'energy_consumption'] = (dataframe.loc[rows_oblique_descending, 'd_horizontal'] * HORIZONTAL_ENERGY +
                                                                  -dataframe.loc[rows_oblique_descending, 'd_vertical'] * DESCENDING_ENERGY)
    return dataframe

# Actually computing energy and time costs for edges

In [200]:
#computing time travel and energy cost for each pair (edges)

nodes_pairs = compute_time_travel(nodes_pairs)

nodes_pairs = compute_energy_cost(nodes_pairs)

nodes_pairs.drop(columns=['d_horizontal','d_vertical', 'euclidean_distance'], inplace=True)

nodes_pairs.head(10)


,source_node,target_node,time_travel,energy_consumption
1,1,2,5.666667,85.000000
3,1,4,5.666667,85.000000
4,1,5,11.000000,550.000000
18,1,19,2.671870,40.078049
19,1,20,0.687184,10.307764
64,2,1,5.666667,85.000000
66,2,3,5.666667,85.000000
69,2,6,11.000000,550.000000
82,2,19,3.004626,45.069391
83,2,20,5.002777,75.041655


# Creating Graph with Networkx


In [201]:
#creating directed graph G from the dataframe of edges with costs

G = nx.from_pandas_edgelist(
    nodes_pairs,                                     # dataframe containing edges
    source='source_node',                            # indicating the start of the edge
    target='target_node',                            # indicating the end of the edge
    edge_attr=['time_travel', 'energy_consumption'], # edge costs
    create_using=nx.DiGraph()                        # creating an oriented graph
)

print(G.edges())

[(1, 2), (1, 4), (1, 5), (1, 19), (1, 20), (2, 1), (2, 3), (2, 6), (2, 19), (2, 20), (4, 1), (4, 3), (4, 8), (5, 1), (5, 6), (5, 8), (5, 11), (5, 12), (19, 1), (19, 2), (19, 18), (19, 20), (19, 23), (20, 1), (20, 2), (20, 17), (20, 19), (20, 24), (3, 2), (3, 4), (3, 7), (6, 2), (6, 5), (6, 7), (6, 9), (7, 3), (7, 6), (7, 8), (7, 10), (8, 4), (8, 5), (8, 7), (8, 11), (8, 12), (11, 5), (11, 8), (11, 10), (11, 12), (11, 15), (12, 5), (12, 8), (12, 9), (12, 11), (12, 16), (9, 6), (9, 10), (9, 12), (9, 13), (10, 7), (10, 9), (10, 11), (10, 14), (13, 9), (13, 14), (13, 16), (14, 10), (14, 13), (14, 15), (15, 11), (15, 14), (15, 16), (16, 12), (16, 13), (16, 15), (17, 18), (17, 20), (17, 21), (17, 50), (17, 51), (17, 60), (17, 63), (17, 64), (18, 17), (18, 19), (18, 22), (18, 50), (18, 51), (18, 57), (18, 61), (18, 62), (21, 17), (21, 22), (21, 24), (21, 27), (21, 34), (21, 35), (21, 39), (21, 54), (21, 55), (21, 57), (21, 59), (21, 60), (21, 61), (21, 62), (21, 63), (21, 64), (50, 17), (50, 

In [202]:
#For example, accessing edge attributes

consume = G[1][2]['energy_consumption'] #energy consumption from node 1 to node 2
time = G[1][2]['time_travel']

print(f"Energy consumption from node 1 to node 2: {consume} J")
print(f"Time travel from node 1 to node 2: {time} s")

Energy consumption from node 1 to node 2: 85.0 J
Time travel from node 1 to node 2: 5.666666666666667 s


In [203]:
# iterating over all the list of tuples of edges, data=True to get edge attributes.
for source, target, costs in G.edges(data=True): 
    print(f"Arco da {source} a {target}:")
    print(f"    Tempo: {costs['time_travel']}")
    print(f"    Energia: {costs['energy_consumption']}")

Arco da 1 a 2:
    Tempo: 5.666666666666667
    Energia: 85.0
Arco da 1 a 4:
    Tempo: 5.666666666666667
    Energia: 85.0
Arco da 1 a 5:
    Tempo: 11.0
    Energia: 550.0
Arco da 1 a 19:
    Tempo: 2.6718699236469
    Energia: 40.078048854703496
Arco da 1 a 20:
    Tempo: 0.6871842709362768
    Energia: 10.307764064044152
Arco da 2 a 1:
    Tempo: 5.666666666666667
    Energia: 85.0
Arco da 2 a 3:
    Tempo: 5.666666666666667
    Energia: 85.0
Arco da 2 a 6:
    Tempo: 11.0
    Energia: 550.0
Arco da 2 a 19:
    Tempo: 3.0046260628866577
    Energia: 45.069390943299865
Arco da 2 a 20:
    Tempo: 5.002777006601211
    Energia: 75.04165509901817
Arco da 4 a 1:
    Tempo: 5.666666666666667
    Energia: 85.0
Arco da 4 a 3:
    Tempo: 5.666666666666667
    Energia: 85.0
Arco da 4 a 8:
    Tempo: 11.0
    Energia: 550.0
Arco da 5 a 1:
    Tempo: 5.5
    Energia: 55.0
Arco da 5 a 6:
    Tempo: 5.666666666666667
    Energia: 85.0
Arco da 5 a 8:
    Tempo: 5.666666666666667
    Energia: 85.0